#  In Class Exercise: Bus Bunching 

###  Bus 36109 "Advanced Decision Modeling with Python", Don Eisenstein
Don Eisenstein &copy; Copyright 2023, University of Chicago 

The Northern State University shuttle makes a simple loop through campus. Transit managers have noticed that their campus shuttle buses are bunching up, leaving big gaps in service.  No matter what they try they the bunching continues ... and the student complaints are piling up.

We build a **simple** simulation to help understand the phenomenon of bus bunching.  

In the first Part there will be no uncertainty.  In the second Part, a small amount of uncertainly will be introduced.

### Part 1

- There are 4 bus stops, 0, 1, 2, 3
- Assume there are 2 buses.  One starts at stop 0, and one at stop 2
- Our unit of time will be seconds
- The travel seconds between each stop is `TIME_BETWEEN_STOPS`
- Customer arrive to each bus stop on average `MEAN_ARRIVALS_PER_STOP_PER_TIME` per second
- The time a bus takes at each stop is `DELAY_TIME_PER_CUSTOMER` per customer awaiting the bus 
- Assume buses have no capacity limit
- We simplify our simulation by assuming no extra delay to drop off passengers.  (Passengers tend to deboard through a rear door as boarding passengers us a front door.)
- Assume all parameters are deterministic
-  **Run the simulation for 50000 seconds. Print out the bus number, bus stop and time at each stop arrival.  The buses should remain perfectly spaced apart.**


### Part 2
- Repeat Part 1 with a small amount of variance, by adding either a 0, 1, or -1 seconds (equally likely) to the travel time between stops.
- **Print out the bus number, bus stop and time at each stop arrival.  You should see buses soon bunch, and stay that way!**
- You can leave your notebook with outputs showing Part 2 results

In [19]:
import numpy as np
import simpy
from simpy_helpers import Entity, Resource, Source, Stats

In [20]:
NUM_BUSES = 2 

# time units in seconds
TIME_BETWEEN_STOPS = 60 * 5  # 300 seconds, or 5 mins
MEAN_ARRIVALS_PER_STOP_PER_TIME = 0.1  # on average 1 every 10 seconds
DELAY_TIME_PER_CUSTOMER = 2   # 2 seconds   

SIM_LENGTH = 50000

In [21]:
# maintain in a simple list the last simulated clock time a bus visited each of the 4 stops
last_visit_time_to_stop = [0,0,0,0]

In [22]:
def delay_time_at_stop(stop_num):
    print(f"env.now: {env.now}  last visit time: {last_visit_time_to_stop[stop_num]}")
    num_arrivals  = (env.now-last_visit_time_to_stop[stop_num]) * MEAN_ARRIVALS_PER_STOP_PER_TIME
    num_arrivals = int(num_arrivals)
    delay_time = num_arrivals * DELAY_TIME_PER_CUSTOMER
    print(f"Stop {stop_num} delay {delay_time}")
    return delay_time


**Define your Resources**

In [23]:
class BusStop0(Resource):
    def service_time(self, entity):
        return delay_time_at_stop(0) 
    
class BusStop1(Resource):
    def service_time(self, entity):
        return delay_time_at_stop(1) 
    
class BusStop2(Resource):
    def service_time(self, entity):
        return delay_time_at_stop(2) 
    
class BusStop3(Resource):
    def service_time(self, entity):
        return delay_time_at_stop(3) 
    
class Drive(Resource):
    def service_time(self, entity):
        return TIME_BETWEEN_STOPS 
        # return TIME_BETWEEN_STOPS + np.random.choice([0,1,-1])


**Define your Source**

In [24]:
class GenerateBuses(Source):
    def interarrival_time(self):
        return 0
    
    def build_entity(self):
        print("Bus generated: ", self.count)
        attributes = {}
        if self.count == 1:
            attributes['next_stop'] = 0 
        else:
            attributes['next_stop'] = 2 
        return Bus(env, attributes) 

**Define your Entity**

In [25]:
#  This is the DRI (Don't Repeat Yourself) version
class Bus(Entity):
    def process(self):
        
        while True: 
            next_stop = self.attributes['next_stop'] 
            print("Time:", env.now, self.name, " Arrive Stop ",next_stop)
            yield self.wait_for_resource(bus_stop[next_stop])
            yield self.process_at_resource(bus_stop[next_stop])
            #  Must set last visit time AFTER you process!
            last_visit_time_to_stop[next_stop] = env.now
            self.release_resource(bus_stop[next_stop])
            
            self.attributes['next_stop'] = (next_stop + 1) % 4
            
            yield self.wait_for_resource(drive)
            yield self.process_at_resource(drive)
            self.release_resource(drive)

**Run your Simulation**

In [26]:
#  This is first detailed version
class Bus(Entity):
    def process(self):
        global last_visit_to_stop 
        
        # while True: 
        while env.now < SIM_LENGTH: 
            # print("Time: ", env.now, " ", self.name)
            # print("Time:", env.now, self.name, self.attributes, " YO")
            
            if self.attributes['next_stop'] == 0:
                print("Time:", env.now, self.name, " Arrive Stop ",self.attributes['next_stop'])
                yield self.wait_for_resource(bus_stop[0])
                yield self.process_at_resource(bus_stop[0])
                #  Must set last visit time AFTER you process!
                last_visit_time_to_stop[self.attributes['next_stop']] = env.now
                self.release_resource(bus_stop[0])

                yield self.wait_for_resource(drive)
                yield self.process_at_resource(drive)
                self.release_resource(drive)
                self.attributes['next_stop'] = 1
                
            if self.attributes['next_stop'] == 1:
                print("Time:", env.now, self.name, " Arrive Stop ",self.attributes['next_stop'])
                yield self.wait_for_resource(bus_stop[1])
                yield self.process_at_resource(bus_stop[1])
                #  Must set last visit time AFTER you process!
                last_visit_time_to_stop[self.attributes['next_stop']] = env.now
                self.release_resource(bus_stop[1])

                yield self.wait_for_resource(drive)
                yield self.process_at_resource(drive)
                self.release_resource(drive)
                self.attributes['next_stop'] = 2
                
            if self.attributes['next_stop'] == 2:
                print("Time:", env.now, self.name, " Arrive Stop ",self.attributes['next_stop'])
                yield self.wait_for_resource(bus_stop[2])
                yield self.process_at_resource(bus_stop[2])
                #  Must set last visit time AFTER you process!
                last_visit_time_to_stop[self.attributes['next_stop']] = env.now
                self.release_resource(bus_stop[2])

                yield self.wait_for_resource(drive)
                yield self.process_at_resource(drive)
                self.release_resource(drive)
                self.attributes['next_stop'] = 3
                
            if self.attributes['next_stop'] == 3:
                print("Time:", env.now, self.name, " Arrive Stop ",self.attributes['next_stop'])
                yield self.wait_for_resource(bus_stop[3])
                yield self.process_at_resource(bus_stop[3])
                #  Must set last visit time AFTER you process!
                last_visit_time_to_stop[self.attributes['next_stop']] = env.now
                self.release_resource(bus_stop[3])

                yield self.wait_for_resource(drive)
                yield self.process_at_resource(drive)
                self.release_resource(drive)
                self.attributes['next_stop'] = 0
                
                


In [27]:
np.random.seed(429) 
env = simpy.Environment()

# Store your BusStop Resource objects in a list
bus_stop = [ 
    BusStop0(env, capacity=1),
    BusStop1(env, capacity=1),
    BusStop2(env, capacity=1), 
    BusStop3(env, capacity=1) 
]

drive = Drive(env, capacity=NUM_BUSES) 

source = GenerateBuses(env, number=NUM_BUSES) 

env.process(source.start(debug=False))
env.run(until=SIM_LENGTH)

Bus generated:  1
Bus generated:  2
Time: 0 Bus 1  Arrive Stop  0
Time: 0 Bus 2  Arrive Stop  2
env.now: 0  last visit time: 0
Stop 0 delay 0
env.now: 0  last visit time: 0
Stop 2 delay 0
Time: 299 Bus 2  Arrive Stop  3
env.now: 299  last visit time: 0
Stop 3 delay 58
Time: 301 Bus 1  Arrive Stop  1
env.now: 301  last visit time: 0
Stop 1 delay 60
Time: 656 Bus 2  Arrive Stop  0
env.now: 656  last visit time: 0
Stop 0 delay 130
Time: 661 Bus 1  Arrive Stop  2
env.now: 661  last visit time: 0
Stop 2 delay 132
Time: 1086 Bus 2  Arrive Stop  1
env.now: 1086  last visit time: 361
Stop 1 delay 144
Time: 1092 Bus 1  Arrive Stop  3
env.now: 1092  last visit time: 357
Stop 3 delay 146
Time: 1529 Bus 2  Arrive Stop  2
env.now: 1529  last visit time: 793
Stop 2 delay 146
Time: 1539 Bus 1  Arrive Stop  0
env.now: 1539  last visit time: 786
Stop 0 delay 150
Time: 1976 Bus 2  Arrive Stop  3
env.now: 1976  last visit time: 1238
Stop 3 delay 146
Time: 1988 Bus 1  Arrive Stop  1
env.now: 1988  last vi